<a href="https://colab.research.google.com/github/ZoeChengYu/ai2026_product3/blob/main/FontDiffuser_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/lctung/fontdiffuser-finetune-colab/blob/main/FontDiffuser_finetuning.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 步驟一： 下載助教的 Github 專案 (Fontdiffuser)

---



In [ ]:
!git clone https://github.com/lctung/fontdiffuser-finetune.git

Cloning into 'fontdiffuser-finetune'...
remote: Enumerating objects: 20332, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 20332 (delta 8), reused 14 (delta 5), pack-reused 20312 (from 1)
Receiving objects: 100% (20332/20332), 63.75 MiB | 21.82 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (20785/20785), done.


In [13]:
!git remote add origin https://github.com/ZoeChengYu/ai2026_product3.git
!git branch -M main
!git push -u origin main

error: remote origin already exists.
fatal: could not read Username for 'https://github.com': No such device or address


# 步驟二：下載預訓練權重檔

---

In [ ]:
%cd /content/fontdiffuser-finetune
# 安裝 gdown（如果尚未安裝）
!pip install -U gdown

# 建立 ckpt 資料夾（如果尚未存在）
!mkdir -p ckpt

# 下載四個檔案
!gdown 1UF4nIcL3PRJeQOQOPFFuGzj30v67rlNn -O ckpt/scr_210000.pth
!gdown 1XIY1QnEIKYmciFnxJ0r48f2LpS8KfZXh -O ckpt/unet.pth
!gdown 1-ywYwsfr8ryE86FgY9Xlub2uhPzZ_WWR -O ckpt/style_encoder.pth
!gdown 1xX-yTNXhniBNR9R5sc1v7-Ghd64HsKIo -O ckpt/content_encoder.pth

# 步驟三：下載並安裝 torch 函式庫

---
### **注意：下面這格程式執行中會需要按一次enter**


In [ ]:
# 安裝 Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils python3.10-dev -y

# 切換默認 python 版本
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.12 1
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 2
!sudo update-alternatives --config python3

# 重新安裝 pip
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

In [ ]:
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117

In [ ]:
import torch
print("GPU 可用:", torch.cuda.is_available())
print("PyTorch 版本:", torch.__version__)

# 步驟四：安裝 requirements.txt 所需套件

---

In [ ]:
!pip install -r /content/fontdiffuser-finetune/requirements.txt

# 步驟五：掛載雲端硬碟空間

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 步驟六：上傳手寫字體壓縮檔&製作訓練資料集合

---

In [8]:
# ----------------上傳手寫字體壓縮檔----------------
%cd /content/fontdiffuser-finetune/split_train_test
from google.colab import files
import zipfile
import os

# 1. 顯示彈跳視窗讓使用者選擇上傳檔案
uploaded = files.upload()

# 2. 假設只上傳一個 zip 檔，取得檔名
for filename in uploaded.keys():
    zip_path = filename
    break
print(filename)

# 3. 解壓縮檔案到目前資料夾
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")  # 解壓縮到目前目錄
print("解壓縮檔案完成")
filename = os.path.splitext(filename)[0]

# ----------------製作訓練資料集合:----------------
%run colab_split_data.py --folder_param "{filename}"

/content/fontdiffuser-finetune/split_train_test


Saving final_dataset.zip to final_dataset.zip
final_dataset.zip
解壓縮檔案完成
共同符合條件的中文檔案數量: 1743
隨機挑選的中文檔案: ['駐', '誅', '宋', '禪', '熠', '光', '過', '遊', '之', '曠', '馨', '衡', '尊', '屯', '楊', '奈', '儼', '應', '維', '詞', '蕩', '困', '八', '致', '鳧', '猷', '唇', '銫', '玖', '湘', '姑', '樓', '高', '宣', '閣', '稷', '歡', '唯', '貴', '肆', '調', '鋦', '銀', '理', '竭', '項', '淙', '紅', '騰', '棄', '壁', '枕', '叛', '兛', '世', '電', '周', '抽', '僕', '勝', '鴛', '刻', '毛', '警', '吐', '左', '接', '南', '恩', '宿', '火', '兄', '惠', '將', '錒', '贊', '厲', '鏌', '性', '噸', '傷', '主', '迤', '屏', '緲', '擲', '留', '猿', '鼓', '獻', '舍', '誠', '鈽', '輔', '鉛', '勸', '戾', '而', '乖', '再', '瓏', '乃', '錫', '盟', '操', '浩', '譽', '鏑', '今', '染', '具', '色', '宴', '絜', '奄', '松', '鐘', '印', '躊', '媧', '房', '落', '零', '馬', '芥', '邑', '嵋', '父', '望', '溴', '特', '累', '張', '扃', '訓', '黜', '友', '冀', '綃', '入', '兝', '盤', '斬', '斤', '鯨', '娜', '尹', '糠', '謙', '遂', '嘆', '螢', '禮', '晦', '制', '名', '乍', '早', '寶', '鈣', '疲', '烹', '梧', '雖', '遠', '盈', '城', '追', '翔', '士', '純', '且', '罪', '閔', '徨', '花', '靡', '共', '侶', '

# 步驟七：開始進行微調FontDiffuser

---

In [10]:
%cd /content/fontdiffuser-finetune
!accelerate launch train.py \
    --seed=123 \
    --experience_name="FontDiffuser_113590051_fineturning" \
    --data_root="data_examples" \
    --output_dir="outputs/FontDiffuser_113590051_fineturning" \
    --report_to="tensorboard" \
    --resolution=96 \
    --style_image_size=96 \
    --content_image_size=96 \
    --content_encoder_downsample_size=3 \
    --channel_attn=True \
    --content_start_channel=64 \
    --style_start_channel=64 \
    --train_batch_size=8 \
    --perceptual_coefficient=0.01 \
    --offset_coefficient=0.5 \
    --max_train_steps=5000 \
    --ckpt_interval=1000 \
    --gradient_accumulation_steps=1 \
    --log_interval=50 \
    --learning_rate=5e-5 \
    --lr_scheduler="linear" \
    --lr_warmup_steps=500 \
    --drop_prob=0.1 \
    --mixed_precision="no" \
    --usecolab

/content/fontdiffuser-finetune
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
pygame 2.6.1 (SDL 2.28.4, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:382: UserWarning: `log_with=tensorboard` was passed but no supported trackers are currently installed.
  warnings.warn(f"`log_with={log_with}` was passed but no supported trackers are currently installed.")
Load the down block  DownBlock2D
Load the down block  MCADownBlock2D
The style_attention cross attention dim in Down Block 1 layer is 1024
The style_attention cross attention dim in Down Block 2 laye